# 🔥 Chapter 5 — Datasets, DataLoaders & Data Pipelines

---

### 🎯 What you'll learn
- `torch.utils.data.Dataset` and `DataLoader`
- Custom datasets for any data format
- Transforms and data augmentation
- Batching, shuffling, and worker processes
- ⚠️ Corner cases: worker memory, reproducibility, imbalanced data

## 5.1 — The Dataset & DataLoader Pattern

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# PyTorch's data loading has two core classes:
# 1. Dataset — defines HOW to access each sample
# 2. DataLoader — wraps Dataset for batching, shuffling, parallel loading

# Let's start with a built-in dataset
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download MNIST
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Sample shape:     {train_dataset[0][0].shape}")
print(f"Label:            {train_dataset[0][1]}")

## 5.2 — Creating a Custom Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SyntheticDataset(Dataset):
    """Custom dataset: y = 2x + 3 + noise"""
    
    def __init__(self, num_samples=1000):
        self.x = torch.randn(num_samples, 1)
        self.y = 2 * self.x + 3 + 0.1 * torch.randn(num_samples, 1)
    
    def __len__(self):
        # REQUIRED: return total number of samples
        return len(self.x)
    
    def __getitem__(self, idx):
        # REQUIRED: return a single sample (features, label)
        return self.x[idx], self.y[idx]

dataset = SyntheticDataset(1000)
print(f"Dataset size: {len(dataset)}")
print(f"Sample: x={dataset[0][0].item():.4f}, y={dataset[0][1].item():.4f}")

## 5.3 — DataLoader: Batching, Shuffling, Workers

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=32,       # Samples per batch
    shuffle=True,        # Shuffle every epoch
    num_workers=0,       # Parallel data loading processes (0 = main process)
    drop_last=False,     # Drop last incomplete batch?
    pin_memory=True,     # Speed up CPU→GPU transfer (use when training on GPU)
)

# Iterate through batches
for batch_idx, (x_batch, y_batch) in enumerate(loader):
    if batch_idx == 0:
        print(f"Batch shape: x={x_batch.shape}, y={y_batch.shape}")
        print(f"First 5 x values: {x_batch[:5, 0].tolist()}")
    break

print(f"Total batches: {len(loader)}")

## 5.4 — ⚠️ Corner Cases

In [ ]:
# ⚠️ Corner Case 1: num_workers > 0 on macOS/Windows
# On macOS, use 'fork' or keep num_workers=0 to avoid crashes.
# On Windows, DataLoader with num_workers > 0 requires:
#   if __name__ == '__main__':   (multiprocessing guard)

# ⚠️ Corner Case 2: Reproducibility
# DataLoader shuffling uses its own RNG. For reproducible results:
def seed_worker(worker_id):
    import numpy as np
    import random
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

reproducible_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    worker_init_fn=seed_worker,
    generator=g,
)

# ⚠️ Corner Case 3: drop_last matters for BatchNorm!
# If last batch has only 1 sample, BatchNorm will crash (needs at least 2).
# Use drop_last=True during training when using BatchNorm.
print("Use drop_last=True with BatchNorm during training!")

In [ ]:
# ⚠️ Corner Case 4: Handling imbalanced datasets
from torch.utils.data import WeightedRandomSampler

# Suppose we have 90% class 0, 10% class 1
labels = torch.cat([torch.zeros(900), torch.ones(100)]).long()
class_counts = torch.bincount(labels).float()
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ⚠️ Cannot use shuffle=True with a custom sampler!
# balanced_loader = DataLoader(dataset, batch_size=32, sampler=sampler)
print(f"Class counts: {class_counts}")
print(f"Weights: {class_weights}")
print("WeightedRandomSampler balances mini-batches!")

## ✏️ Exercises

In [ ]:
# Exercise 1: Create a custom Dataset for the XOR problem:
# Inputs: (0,0), (0,1), (1,0), (1,1)  Labels: 0, 1, 1, 0
# Then wrap in a DataLoader with batch_size=2 and iterate.
# YOUR CODE HERE


# Exercise 2: Create a Dataset that reads lines from a text file.
# Each line has format "feature1,feature2,label".
# Write the CSV file first, then create the Dataset.
# YOUR CODE HERE
